# CSCI 6970 – Data Science MS Project

## Data Collection and Integration

**Project Title:** Classifying Diagnosed Diabetes Using Demographic, Lifestyle, and Health Indicators from NHANES Data

**Dataset:** NHANES August 2021 – August 2023

**Data Source:** National Health and Nutrition Examination Survey (NHANES), Centers for Disease Control and Prevention (CDC)

**Input Data:** Eight NHANES datasets (DEMO_L, DIQ_L, BMX_L, BPXO_L, BPQ_L, PAQ_L, SMQ_L, ALQ_L)

**Output Dataset:** nhanes_diabetes_integrated.csv

This notebook focuses on collecting, inspecting, and integrating NHANES datasets using participant identifiers (SEQN) to create a unified dataset for subsequent data cleaning, exploratory data analysis, and machine learning.

## 1. Project Environment Setup

### 1.1 Project Overview

This project aims to develop machine learning classification models to identify diagnosed diabetes using demographic, lifestyle, and health indicators from the National Health and Nutrition Examination Survey (NHANES) August 2021–August 2023 dataset.

The project uses Python, Pandas, NumPy, Scikit-learn, and JupyterLab for data collection, preprocessing, statistical analysis, and machine learning.

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nCSCI6970 Environment Ready!")

Python: 3.9.6 (default, Aug 25 2026, 21:26:21) 
[Clang 21.0.0 (clang-2100.3.34.2)]
Pandas: 2.3.3
NumPy: 2.0.2
Scikit-learn: 1.6.1

CSCI6970 Environment Ready!


### 1.2 Project Directory Structure

The project is organized into separate directories for raw data, processed data, notebooks, outputs, and scripts to maintain a reproducible workflow.

In [2]:
from pathlib import Path

# Project directory
PROJECT_DIR = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

Project directory: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970
Raw data directory: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970/data/raw
Processed data directory: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970/data/processed


## 2. NHANES Data Collection

### 2.1 NHANES Dataset Selection

The study uses publicly available NHANES August 2021–August 2023 data released by the Centers for Disease Control and Prevention (CDC).

### 2.2 Identification of Required NHANES Datasets

Eight NHANES datasets were selected for this study: demographics, diabetes, body measurements, blood pressure examination, blood pressure questionnaire, physical activity, smoking, and alcohol use.

### 2.3 Downloading NHANES XPT Files

In [3]:
import requests
from pathlib import Path
import time

# NHANES August 2021–August 2023
BASE_URL = (
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/"
    "Public/2021/DataFiles/"
)

DATA_FILES = [
    "DEMO_L",
    "DIQ_L",
    "BMX_L",
    "BPXO_L",
    "BPQ_L",
    "PAQ_L",
    "SMQ_L",
    "ALQ_L"
]

RAW_DIR.mkdir(parents=True, exist_ok=True)

download_results = []

for file_name in DATA_FILES:

    file_url = f"{BASE_URL}{file_name}.XPT"
    file_path = RAW_DIR / f"{file_name}.XPT"

    try:
        print(f"Downloading {file_name}...", end=" ")

        response = requests.get(
            file_url,
            timeout=120
        )

        response.raise_for_status()

        # Save original XPT file
        file_path.write_bytes(response.content)

        file_size = file_path.stat().st_size / 1024

        download_results.append({
            "Dataset": file_name,
            "Status": "Downloaded",
            "Size (KB)": round(file_size, 2)
        })

        print(f"Success ({file_size:.2f} KB)")

    except Exception as error:

        download_results.append({
            "Dataset": file_name,
            "Status": "Failed",
            "Size (KB)": 0
        })

        print(f"Failed: {error}")

    time.sleep(1)

# Display download summary
import pandas as pd

download_summary = pd.DataFrame(download_results)

display(download_summary)

/Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,Dataset,Status,Size (KB)
0,DEMO_L,Downloaded,2521.64
1,DIQ_L,Downloaded,827.73
2,BMX_L,Downloaded,1526.56
3,BPXO_L,Downloaded,680.39
4,BPQ_L,Downloaded,400.08
5,PAQ_L,Downloaded,399.92
6,SMQ_L,Downloaded,635.86
7,ALQ_L,Downloaded,447.58


### 2.4 Raw Data Storage and Verification

The downloaded XPT files are stored in the data/raw directory. Their presence and file sizes are verified to confirm successful data collection.

In [4]:
print("NHANES RAW DATA VERIFICATION")
print("-" * 45)

for file in sorted(RAW_DIR.glob("*.XPT")):
    print(
        f"{file.name:15} "
        f"{file.stat().st_size / 1024:.2f} KB"
    )

print("-" * 45)
print("Total files:", len(list(RAW_DIR.glob("*.XPT"))))

NHANES RAW DATA VERIFICATION
---------------------------------------------
ALQ_L.XPT       447.58 KB
BMX_L.XPT       1526.56 KB
BPQ_L.XPT       400.08 KB
BPXO_L.XPT      680.39 KB
DEMO_L.XPT      2521.64 KB
DIQ_L.XPT       827.73 KB
PAQ_L.XPT       399.92 KB
SMQ_L.XPT       635.86 KB
---------------------------------------------
Total files: 8


## 3. NHANES Dataset Inspection

### 3.1 Loading NHANES XPT Files

The downloaded NHANES datasets are loaded using Pandas to examine their structure.

### 3.2 Inspecting Dataset Dimensions and Variables

The number of rows, columns, and available variables is examined for each dataset.

### 3.3 Verifying Participant Identifier (SEQN)

The SEQN variable is checked across all datasets because it serves as the common participant identifier for subsequent data integration.

In [5]:
import pandas as pd
from pathlib import Path

print("NHANES DATASET INSPECTION")
print("=" * 60)

dataset_summary = []

for file_path in sorted(RAW_DIR.glob("*.XPT")):
    df = pd.read_sas(file_path, format="xport")

    dataset_summary.append({
        "Dataset": file_path.stem,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "SEQN Available": "SEQN" in df.columns
    })

    print(f"\n{file_path.stem}")
    print("Shape:", df.shape)
    print("First 10 columns:", list(df.columns[:10]))

summary_df = pd.DataFrame(dataset_summary)

print("\n" + "=" * 60)
display(summary_df)

NHANES DATASET INSPECTION

ALQ_L
Shape: (6337, 9)
First 10 columns: ['SEQN', 'ALQ111', 'ALQ121', 'ALQ130', 'ALQ142', 'ALQ270', 'ALQ280', 'ALQ151', 'ALQ170']

BMX_L
Shape: (8860, 22)
First 10 columns: ['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT']

BPQ_L
Shape: (8501, 6)
First 10 columns: ['SEQN', 'BPQ020', 'BPQ030', 'BPQ150', 'BPQ080', 'BPQ101D']

BPXO_L
Shape: (7801, 12)
First 10 columns: ['SEQN', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1']

DEMO_L
Shape: (11933, 27)
First 10 columns: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM']

DIQ_L
Shape: (11744, 9)
First 10 columns: ['SEQN', 'DIQ010', 'DID040', 'DIQ160', 'DIQ180', 'DIQ050', 'DID060', 'DIQ060U', 'DIQ070']

PAQ_L
Shape: (8153, 8)
First 10 columns: ['SEQN', 'PAD790Q', 'PAD790U', 'PAD800', 'PAD810Q', 'PAD810U', 'PAD820', 'PAD680']

SMQ_L
Shape: (9015, 9

,Dataset,Rows,Columns,SEQN Available
0,ALQ_L,6337,9,True
1,BMX_L,8860,22,True
2,BPQ_L,8501,6,True
3,BPXO_L,7801,12,True
4,DEMO_L,11933,27,True
5,DIQ_L,11744,9,True
6,PAQ_L,8153,8,True
7,SMQ_L,9015,9,True


## 4. Diabetes Target Variable Preparation

### 4.1 Loading the Diabetes Questionnaire Dataset

The DIQ_L dataset is loaded to identify the variable used as the diabetes classification target.

### 4.2 Identification of Diabetes Target Variable (DIQ010)

DIQ010 records whether a participant has been told by a doctor or health professional that they have diabetes.

### 4.3 Inspecting Diabetes Response Categories

The response counts and percentages are examined, including valid, borderline, refused, and missing responses.

In [6]:
# Load the diabetes questionnaire dataset
diq_df = pd.read_sas(
    RAW_DIR / "DIQ_L.XPT",
    format="xport"
)

# Check the target variable
print("DIABETES TARGET VARIABLE: DIQ010")
print("=" * 50)

print("\nResponse counts:")
print(diq_df["DIQ010"].value_counts(dropna=False).sort_index())

print("\nResponse percentages:")
print(
    (diq_df["DIQ010"].value_counts(normalize=True, dropna=False) * 100)
    .sort_index()
    .round(2)
)

print("\nMissing target values:")
print(diq_df["DIQ010"].isna().sum())

DIABETES TARGET VARIABLE: DIQ010

Response counts:
DIQ010
1.0     1081
2.0    10371
3.0      284
9.0        4
NaN        4
Name: count, dtype: int64

Response percentages:
DIQ010
1.0     9.20
2.0    88.31
3.0     2.42
9.0     0.03
NaN     0.03
Name: proportion, dtype: float64

Missing target values:
4


### 4.4 Creating the Binary Diabetes Target Variable

The binary target variable DIABETES is created using the following coding:

- 1 = Yes, diagnosed with diabetes.
- 0 = No, not diagnosed with diabetes.

Borderline, refused, and missing target responses are excluded from the primary binary classification dataset.

### 4.5 Examining Target Distribution and Duplicate IDs

The resulting class distribution and participant identifier uniqueness are verified.

In [7]:
# Create a copy of the diabetes questionnaire dataset
diabetes_df = diq_df.copy()

# Keep only valid Yes/No responses
diabetes_df = diabetes_df[
    diabetes_df["DIQ010"].isin([1.0, 2.0])
].copy()

# Create binary target variable
# 1 = Yes, diagnosed with diabetes
# 0 = No, not diagnosed with diabetes
diabetes_df["DIABETES"] = diabetes_df["DIQ010"].map({
    1.0: 1,
    2.0: 0
})

# Display target distribution
print("BINARY DIABETES TARGET SUMMARY")
print("=" * 45)

print("Total participants:", len(diabetes_df))
print("\nTarget distribution:")
print(diabetes_df["DIABETES"].value_counts().sort_index())

print("\nTarget percentages:")
print(
    (diabetes_df["DIABETES"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

print("\nDuplicate participant IDs:", diabetes_df["SEQN"].duplicated().sum())

# Keep only participant ID and target variable
diabetes_target = diabetes_df[["SEQN", "DIABETES"]].copy()

display(diabetes_target.head())

BINARY DIABETES TARGET SUMMARY
Total participants: 11452

Target distribution:
DIABETES
0    10371
1     1081
Name: count, dtype: int64

Target percentages:
DIABETES
0    90.56
1     9.44
Name: proportion, dtype: float64

Duplicate participant IDs: 0


,SEQN,DIABETES
0,130378.0,0
1,130379.0,0
2,130380.0,1
3,130381.0,0
4,130382.0,0


## 5. Adult Population Filtering and Initial Data Merging

### 5.1 Loading the Demographic Dataset

The DEMO_L dataset is loaded to obtain participant demographic information.

### 5.2 Inspecting Age Variable (RIDAGEYR)

The age variable is examined using descriptive statistics and missing-value checks.

### 5.3 Filtering Adult Participants (Age 20 and Above)

The study population is restricted to participants aged 20 years and older.

In [8]:
# Load NHANES demographic dataset
demo_df = pd.read_sas(
    RAW_DIR / "DEMO_L.XPT",
    format="xport"
)

print("DEMOGRAPHIC DATA INSPECTION")
print("=" * 45)

print("Dataset shape:", demo_df.shape)

print("\nAge variable information:")
print(demo_df["RIDAGEYR"].describe())

print("\nMissing age values:")
print(demo_df["RIDAGEYR"].isna().sum())

# Count participants aged 20 or older
adult_demo = demo_df[
    demo_df["RIDAGEYR"] >= 20
].copy()

print("\nParticipants aged 20 or older:", len(adult_demo))

# Check participant ID uniqueness
print(
    "Duplicate SEQN IDs:",
    adult_demo["SEQN"].duplicated().sum()
)

DEMOGRAPHIC DATA INSPECTION
Dataset shape: (11933, 27)

Age variable information:
count    1.193300e+04
mean     3.831786e+01
std      2.560199e+01
min      5.397605e-79
25%      1.300000e+01
50%      3.700000e+01
75%      6.200000e+01
max      8.000000e+01
Name: RIDAGEYR, dtype: float64

Missing age values:
0

Participants aged 20 or older: 7809
Duplicate SEQN IDs: 0


### 5.4 Merging Adult Demographics with Diabetes Target

The adult demographic dataset is merged with the binary diabetes target using SEQN. An inner join retains participants with both demographic information and a valid diabetes response.

### 5.5 Verifying the Adult Diabetes Dataset

The merged dataset dimensions, diabetes distribution, age range, and duplicate participant IDs are examined.

In [9]:
# Merge adult demographics with diabetes target
adult_diabetes_df = pd.merge(
    adult_demo,
    diabetes_target,
    on="SEQN",
    how="inner",
    validate="one_to_one"
)

print("ADULT DIABETES DATASET")
print("=" * 50)

print("Dataset shape:", adult_diabetes_df.shape)
print("Total adult participants:", len(adult_diabetes_df))

print("\nDiabetes distribution:")
print(adult_diabetes_df["DIABETES"].value_counts().sort_index())

print("\nDiabetes percentages:")
print(
    (adult_diabetes_df["DIABETES"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

print("\nAge range:")
print(
    adult_diabetes_df["RIDAGEYR"].min(),
    "to",
    adult_diabetes_df["RIDAGEYR"].max()
)

print("\nDuplicate participant IDs:",
      adult_diabetes_df["SEQN"].duplicated().sum())

display(adult_diabetes_df[["SEQN", "RIDAGEYR", "DIABETES"]].head())

ADULT DIABETES DATASET
Dataset shape: (7536, 28)
Total adult participants: 7536

Diabetes distribution:
DIABETES
0    6466
1    1070
Name: count, dtype: int64

Diabetes percentages:
DIABETES
0    85.8
1    14.2
Name: proportion, dtype: float64

Age range:
20.0 to 80.0

Duplicate participant IDs: 0


,SEQN,RIDAGEYR,DIABETES
0,130378.0,43.0,0
1,130379.0,66.0,0
2,130380.0,44.0,1
3,130384.0,43.0,0
4,130385.0,65.0,0


## 6. Additional NHANES Dataset Integration

### 6.1 Integrating Body Measurements (BMX_L)

The BMX_L dataset contains body measurement information collected during NHANES examinations.

For this project, we will examine variables such as:

- BMXBMI – Body Mass Index (BMI)
- BMXWT – Weight measurement
- BMXHT – Standing height
- BMXWAIST – Waist circumference

These variables will be considered as potential predictors of diagnosed diabetes.

The dataset will be integrated with the existing adult diabetes dataset using the participant identifier SEQN.

In [15]:
# Load NHANES body measurements dataset
bmx_df = pd.read_sas(
    RAW_DIR / "BMX_L.XPT",
    format="xport"
)

print("BODY MEASUREMENTS DATASET INSPECTION")
print("=" * 55)

print("Dataset shape:", bmx_df.shape)

print("\nAvailable variables:")
print(bmx_df.columns.tolist())

print("\nSelected body measurement variables:")

bmx_variables = [
    "SEQN",
    "BMXBMI",
    "BMXWT",
    "BMXHT",
    "BMXWAIST"
]

display(bmx_df[bmx_variables].head())

print("\nMissing values in selected variables:")
display(bmx_df[bmx_variables].isna().sum())

print(
    "\nDuplicate participant IDs:",
    bmx_df["SEQN"].duplicated().sum()
)

BODY MEASUREMENTS DATASET INSPECTION
Dataset shape: (8860, 22)

Available variables:
['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXHIP', 'BMIHIP']

Selected body measurement variables:


,SEQN,BMXBMI,BMXWT,BMXHT,BMXWAIST
0,130378.0,27.0,86.9,179.5,98.3
1,130379.0,33.5,101.8,174.2,114.7
2,130380.0,29.7,69.4,152.9,93.5
3,130381.0,23.8,34.3,120.1,70.4
4,130382.0,NaN,13.6,NaN,NaN



Missing values in selected variables:


SEQN          0
BMXBMI      389
BMXWT       106
BMXHT       361
BMXWAIST    670
dtype: int64


Duplicate participant IDs: 0


### 6.1.2 Restoring Project Variables After Kernel Restart

The Jupyter kernel was restarted, resulting in the loss of previously loaded variables.

The required project paths, libraries, and NHANES datasets are reloaded from the existing local project directory to continue the data collection process.

The original downloaded datasets remain unchanged.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Raw data folder exists:", RAW_DIR.exists())

Project directory: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970
Raw data directory: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970/data/raw
Raw data folder exists: True


### 6.1.3 Reloading Required NHANES Datasets

The demographic, diabetes questionnaire, and body measurement datasets are reloaded from the local raw data directory.

These datasets are required to reconstruct the adult diabetes dataset and continue the body measurement integration process.

In [4]:
demo_df = pd.read_sas(
    RAW_DIR / "DEMO_L.XPT",
    format="xport"
)

diq_df = pd.read_sas(
    RAW_DIR / "DIQ_L.XPT",
    format="xport"
)

bmx_df = pd.read_sas(
    RAW_DIR / "BMX_L.XPT",
    format="xport"
)

print("DEMO_L:", demo_df.shape)
print("DIQ_L:", diq_df.shape)
print("BMX_L:", bmx_df.shape)

DEMO_L: (11933, 27)
DIQ_L: (11744, 9)
BMX_L: (8860, 22)


### 6.1.4 Recreating the Binary Diabetes Target

The binary diabetes target variable is reconstructed using the DIQ010 variable from the NHANES diabetes questionnaire.

Only valid Yes/No responses are retained.

- 1 = Diagnosed diabetes
- 0 = No diagnosed diabetes

Borderline, refused, and missing responses are excluded from the primary binary classification dataset.

In [5]:
diabetes_df = diq_df[
    diq_df["DIQ010"].isin([1.0, 2.0])
].copy()

diabetes_df["DIABETES"] = (
    diabetes_df["DIQ010"].map({
        1.0: 1,
        2.0: 0
    })
)

diabetes_target = diabetes_df[
    ["SEQN", "DIABETES"]
].copy()

print(diabetes_target["DIABETES"].value_counts())

DIABETES
0    10371
1     1081
Name: count, dtype: int64


### 6.1.5 Recreating the Adult Diabetes Dataset

The demographic dataset is filtered to include participants aged 20 years and older.

The filtered demographic records are merged with the binary diabetes target using the SEQN participant identifier.

An inner join is applied to retain participants with valid demographic information and diabetes responses.

In [6]:
adult_demo = demo_df[
    demo_df["RIDAGEYR"] >= 20
].copy()

adult_diabetes_df = pd.merge(
    adult_demo,
    diabetes_target,
    on="SEQN",
    how="inner",
    validate="one_to_one"
)

print("ADULT DIABETES DATASET RESTORED")
print("Dataset shape:", adult_diabetes_df.shape)
print("Participants:", len(adult_diabetes_df))
print(adult_diabetes_df["DIABETES"].value_counts())

ADULT DIABETES DATASET RESTORED
Dataset shape: (7536, 28)
Participants: 7536
DIABETES
0    6466
1    1070
Name: count, dtype: int64


### 6.1.6 Merging Body Measurements with Adult Diabetes Dataset

The selected body measurement variables from BMX_L are integrated with the existing adult diabetes dataset using the SEQN participant identifier.

A left join is used to preserve all participants in the adult diabetes dataset, including participants with unavailable body measurements.

The merged dataset is evaluated using participant counts, merge status, missing values, and duplicate identifier checks.

In [7]:
# Select relevant body measurement variables
bmx_selected = bmx_df[
    [
        "SEQN",
        "BMXBMI",
        "BMXWT",
        "BMXHT",
        "BMXWAIST"
    ]
].copy()

# Merge body measurements with adult diabetes dataset
adult_diabetes_bmx_df = pd.merge(
    adult_diabetes_df,
    bmx_selected,
    on="SEQN",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("BODY MEASUREMENTS MERGE SUMMARY")
print("=" * 55)

print(
    "Original adult diabetes participants:",
    len(adult_diabetes_df)
)

print(
    "Merged dataset participants:",
    len(adult_diabetes_bmx_df)
)

print("\nMerge status:")
print(adult_diabetes_bmx_df["_merge"].value_counts())

# Remove temporary merge indicator
adult_diabetes_bmx_df.drop(
    columns=["_merge"],
    inplace=True
)

print("\nMerged dataset shape:")
print(adult_diabetes_bmx_df.shape)

print("\nDuplicate participant IDs:")
print(adult_diabetes_bmx_df["SEQN"].duplicated().sum())

print("\nMissing body measurement values:")
display(
    adult_diabetes_bmx_df[
        ["BMXBMI", "BMXWT", "BMXHT", "BMXWAIST"]
    ].isna().sum()
)

print("\nMerged dataset preview:")
display(
    adult_diabetes_bmx_df[
        [
            "SEQN",
            "RIDAGEYR",
            "DIABETES",
            "BMXBMI",
            "BMXWT",
            "BMXHT",
            "BMXWAIST"
        ]
    ].head()
)

BODY MEASUREMENTS MERGE SUMMARY
Original adult diabetes participants: 7536
Merged dataset participants: 7536

Merge status:
_merge
both          5843
left_only     1693
right_only       0
Name: count, dtype: int64

Merged dataset shape:
(7536, 32)

Duplicate participant IDs:
0

Missing body measurement values:


BMXBMI      1784
BMXWT       1771
BMXHT       1757
BMXWAIST    1983
dtype: int64


Merged dataset preview:


,SEQN,RIDAGEYR,DIABETES,BMXBMI,BMXWT,BMXHT,BMXWAIST
0,130378.0,43.0,0,27.0,86.9,179.5,98.3
1,130379.0,66.0,0,33.5,101.8,174.2,114.7
2,130380.0,44.0,1,29.7,69.4,152.9,93.5
3,130384.0,43.0,0,NaN,NaN,NaN,NaN
4,130385.0,65.0,0,NaN,NaN,NaN,NaN


### 6.2 Blood Pressure Examination Data (BPXO_L)

This section examines blood pressure measurements collected during the NHANES examination. The BPXO_L dataset contains systolic and diastolic blood pressure readings.

The participant identifier (SEQN) will be used to integrate the selected blood pressure variables into the adult diabetes dataset.

In [8]:
# Load blood pressure examination dataset
bpxo_df = pd.read_sas(RAW_DIR / "BPXO_L.XPT", format="xport")

print("BLOOD PRESSURE EXAMINATION DATA")
print("=" * 50)

print("Dataset shape:", bpxo_df.shape)

print("\nAvailable columns:")
print(bpxo_df.columns.tolist())

print("\nSelected blood pressure variables:")
bpxo_columns = [
    col for col in bpxo_df.columns
    if col == "SEQN" or col.startswith("BPX")
]
print(bpxo_columns)

print("\nMissing values in selected variables:")
display(bpxo_df[bpxo_columns].isna().sum())

print("\nDuplicate participant IDs:")
print(bpxo_df["SEQN"].duplicated().sum())

print("\nDataset preview:")
display(bpxo_df[bpxo_columns].head())

BLOOD PRESSURE EXAMINATION DATA
Dataset shape: (7801, 12)

Available columns:
['SEQN', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3']

Selected blood pressure variables:
['SEQN', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3']

Missing values in selected variables:


SEQN          0
BPXOSY1     284
BPXODI1     284
BPXOSY2     296
BPXODI2     296
BPXOSY3     321
BPXODI3     321
BPXOPLS1    284
BPXOPLS2    296
BPXOPLS3    321
dtype: int64


Duplicate participant IDs:
0

Dataset preview:


,SEQN,BPXOSY1,BPXODI1,BPXOSY2,BPXODI2,BPXOSY3,BPXODI3,BPXOPLS1,BPXOPLS2,BPXOPLS3
0,130378.0,135.0,98.0,131.0,96.0,132.0,94.0,82.0,79.0,82.0
1,130379.0,121.0,84.0,117.0,76.0,113.0,76.0,72.0,71.0,73.0
2,130380.0,111.0,79.0,112.0,80.0,104.0,76.0,84.0,83.0,77.0
3,130386.0,110.0,72.0,120.0,74.0,115.0,75.0,59.0,64.0,64.0
4,130387.0,143.0,76.0,136.0,74.0,145.0,78.0,80.0,80.0,77.0


### 6.2.1 Merging Blood Pressure Examination Data

The selected blood pressure examination variables are merged with the adult diabetes dataset using the participant identifier (SEQN).

A left join is used to preserve all participants in the existing dataset. Missing blood pressure measurements will be retained for handling during the data-cleaning stage.

In [9]:
# Select blood pressure examination variables
bpxo_selected = bpxo_df[bpxo_columns].copy()

# Merge blood pressure data with the existing dataset
adult_diabetes_bpxo_df = pd.merge(
    adult_diabetes_bmx_df,
    bpxo_selected,
    on="SEQN",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("BLOOD PRESSURE MERGE SUMMARY")
print("=" * 55)

print("Original participants:", len(adult_diabetes_bmx_df))
print("Merged participants:", len(adult_diabetes_bpxo_df))

print("\nMerge status:")
print(adult_diabetes_bpxo_df["_merge"].value_counts())

# Remove merge indicator
adult_diabetes_bpxo_df.drop(columns=["_merge"], inplace=True)

print("\nFinal dataset shape:", adult_diabetes_bpxo_df.shape)

print("\nDuplicate participant IDs:")
print(adult_diabetes_bpxo_df["SEQN"].duplicated().sum())

print("\nMissing blood pressure values:")
display(adult_diabetes_bpxo_df[bpxo_columns[1:]].isna().sum())

print("\nMerged dataset preview:")
display(adult_diabetes_bpxo_df[
    ["SEQN", "RIDAGEYR", "DIABETES",
     "BPXOSY1", "BPXODI1", "BPXOSY2", "BPXODI2"]
].head())

BLOOD PRESSURE MERGE SUMMARY
Original participants: 7536
Merged participants: 7536

Merge status:
_merge
both          5843
left_only     1693
right_only       0
Name: count, dtype: int64

Final dataset shape: (7536, 41)

Duplicate participant IDs:
0

Missing blood pressure values:


BPXOSY1     1885
BPXODI1     1885
BPXOSY2     1892
BPXODI2     1892
BPXOSY3     1905
BPXODI3     1905
BPXOPLS1    1885
BPXOPLS2    1892
BPXOPLS3    1905
dtype: int64


Merged dataset preview:


,SEQN,RIDAGEYR,DIABETES,BPXOSY1,BPXODI1,BPXOSY2,BPXODI2
0,130378.0,43.0,0,135.0,98.0,131.0,96.0
1,130379.0,66.0,0,121.0,84.0,117.0,76.0
2,130380.0,44.0,1,111.0,79.0,112.0,80.0
3,130384.0,43.0,0,NaN,NaN,NaN,NaN
4,130385.0,65.0,0,NaN,NaN,NaN,NaN


### 6.3 Blood Pressure Questionnaire Data (BPQ_L)

This section examines self-reported blood pressure information from the NHANES questionnaire dataset.

The dataset will be inspected to identify relevant variables related to participants' history of high blood pressure. The participant identifier (SEQN) will be used to integrate the selected variables into the existing adult diabetes dataset.

In [10]:
# Load blood pressure questionnaire dataset
bpq_df = pd.read_sas(RAW_DIR / "BPQ_L.XPT", format="xport")

print("BLOOD PRESSURE QUESTIONNAIRE DATA")
print("=" * 50)

print("Dataset shape:", bpq_df.shape)

print("\nAvailable columns:")
print(bpq_df.columns.tolist())

print("\nMissing values:")
display(bpq_df.isna().sum())

print("\nDuplicate participant IDs:")
print(bpq_df["SEQN"].duplicated().sum())

print("\nDataset preview:")
display(bpq_df.head())

BLOOD PRESSURE QUESTIONNAIRE DATA
Dataset shape: (8501, 6)

Available columns:
['SEQN', 'BPQ020', 'BPQ030', 'BPQ150', 'BPQ080', 'BPQ101D']

Missing values:


SEQN          0
BPQ020        3
BPQ030     5533
BPQ150     5532
BPQ080        3
BPQ101D       3
dtype: int64


Duplicate participant IDs:
0

Dataset preview:


,SEQN,BPQ020,BPQ030,BPQ150,BPQ080,BPQ101D
0,130378.0,1.0,1.0,1.0,2.0,2.0
1,130379.0,1.0,1.0,1.0,2.0,2.0
2,130380.0,2.0,NaN,NaN,1.0,1.0
3,130384.0,2.0,NaN,NaN,2.0,2.0
4,130385.0,2.0,NaN,NaN,2.0,2.0


### 6.3.1 Merging Blood Pressure Questionnaire Data

The selected blood pressure questionnaire variables are merged with the existing adult diabetes dataset using the participant identifier (SEQN).

A left join is used to preserve all participants. Missing questionnaire responses are retained for handling during the data-cleaning stage.

In [11]:
# Select blood pressure questionnaire variables
bpq_selected = bpq_df.copy()

# Merge questionnaire data with the existing dataset
adult_diabetes_bpq_df = pd.merge(
    adult_diabetes_bpxo_df,
    bpq_selected,
    on="SEQN",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("BLOOD PRESSURE QUESTIONNAIRE MERGE SUMMARY")
print("=" * 55)

print("Original participants:", len(adult_diabetes_bpxo_df))
print("Merged participants:", len(adult_diabetes_bpq_df))

print("\nMerge status:")
print(adult_diabetes_bpq_df["_merge"].value_counts())

# Remove merge indicator
adult_diabetes_bpq_df.drop(columns=["_merge"], inplace=True)

print("\nFinal dataset shape:", adult_diabetes_bpq_df.shape)

print("\nDuplicate participant IDs:")
print(adult_diabetes_bpq_df["SEQN"].duplicated().sum())

print("\nMissing questionnaire values:")
display(adult_diabetes_bpq_df[bpq_selected.columns.drop("SEQN")].isna().sum())

print("\nMerged dataset preview:")
display(adult_diabetes_bpq_df[
    ["SEQN", "RIDAGEYR", "DIABETES",
     "BPQ020", "BPQ030", "BPQ150", "BPQ080", "BPQ101D"]
].head())

BLOOD PRESSURE QUESTIONNAIRE MERGE SUMMARY
Original participants: 7536
Merged participants: 7536

Merge status:
_merge
both          7536
left_only        0
right_only       0
Name: count, dtype: int64

Final dataset shape: (7536, 46)

Duplicate participant IDs:
0

Missing questionnaire values:


BPQ020        0
BPQ030     4759
BPQ150     4758
BPQ080        0
BPQ101D       0
dtype: int64


Merged dataset preview:


,SEQN,RIDAGEYR,DIABETES,BPQ020,BPQ030,BPQ150,BPQ080,BPQ101D
0,130378.0,43.0,0,1.0,1.0,1.0,2.0,2.0
1,130379.0,66.0,0,1.0,1.0,1.0,2.0,2.0
2,130380.0,44.0,1,2.0,NaN,NaN,1.0,1.0
3,130384.0,43.0,0,2.0,NaN,NaN,2.0,2.0
4,130385.0,65.0,0,2.0,NaN,NaN,2.0,2.0


### 6.4 Physical Activity Questionnaire Data (PAQ_L)

This section examines physical activity information collected through the NHANES questionnaire.

The dataset will be inspected to identify relevant variables related to physical activity, recreational activities, and sedentary behavior. The participant identifier (SEQN) will be used to integrate the selected variables into the existing adult diabetes dataset.

In [12]:
# Load physical activity questionnaire dataset
paq_df = pd.read_sas(RAW_DIR / "PAQ_L.XPT", format="xport")

print("PHYSICAL ACTIVITY QUESTIONNAIRE DATA")
print("=" * 50)

print("Dataset shape:", paq_df.shape)

print("\nAvailable columns:")
print(paq_df.columns.tolist())

print("\nMissing values:")
display(paq_df.isna().sum())

print("\nDuplicate participant IDs:")
print(paq_df["SEQN"].duplicated().sum())

print("\nDataset preview:")
display(paq_df.head())

PHYSICAL ACTIVITY QUESTIONNAIRE DATA
Dataset shape: (8153, 8)

Available columns:
['SEQN', 'PAD790Q', 'PAD790U', 'PAD800', 'PAD810Q', 'PAD810U', 'PAD820', 'PAD680']

Missing values:


SEQN          0
PAD790Q      18
PAD790U       0
PAD800     1763
PAD810Q      14
PAD810U       0
PAD820     4466
PAD680       15
dtype: int64


Duplicate participant IDs:
0

Dataset preview:


,SEQN,PAD790Q,PAD790U,PAD800,PAD810Q,PAD810U,PAD820,PAD680
0,130378.0,3.000000e+00,b'W',45.0,3.000000e+00,b'W',45.0,360.0
1,130379.0,4.000000e+00,b'W',45.0,3.000000e+00,b'W',45.0,480.0
2,130380.0,1.000000e+00,b'W',20.0,5.397605e-79,b'',NaN,240.0
3,130384.0,5.397605e-79,b'',NaN,5.397605e-79,b'',NaN,60.0
4,130385.0,1.000000e+00,b'D',90.0,1.000000e+00,b'W',60.0,180.0


### 6.4.1 Merging Physical Activity Questionnaire Data

The physical activity questionnaire variables are merged with the existing adult diabetes dataset using the participant identifier (SEQN).

A left join is used to preserve all participants. Missing physical activity responses are retained for further investigation and handling during the data-cleaning stage.

In [13]:
# Select physical activity questionnaire variables
paq_selected = paq_df.copy()

# Merge physical activity data with the existing dataset
adult_diabetes_paq_df = pd.merge(
    adult_diabetes_bpq_df,
    paq_selected,
    on="SEQN",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("PHYSICAL ACTIVITY MERGE SUMMARY")
print("=" * 55)

print("Original participants:", len(adult_diabetes_bpq_df))
print("Merged participants:", len(adult_diabetes_paq_df))

print("\nMerge status:")
print(adult_diabetes_paq_df["_merge"].value_counts())

# Remove merge indicator
adult_diabetes_paq_df.drop(columns=["_merge"], inplace=True)

print("\nFinal dataset shape:", adult_diabetes_paq_df.shape)

print("\nDuplicate participant IDs:")
print(adult_diabetes_paq_df["SEQN"].duplicated().sum())

print("\nMissing physical activity values:")
display(adult_diabetes_paq_df[paq_selected.columns.drop("SEQN")].isna().sum())

print("\nMerged dataset preview:")
display(adult_diabetes_paq_df[
    ["SEQN", "RIDAGEYR", "DIABETES",
     "PAD790Q", "PAD790U", "PAD800",
     "PAD810Q", "PAD810U", "PAD820", "PAD680"]
].head())

PHYSICAL ACTIVITY MERGE SUMMARY
Original participants: 7536
Merged participants: 7536

Merge status:
_merge
both          7536
left_only        0
right_only       0
Name: count, dtype: int64

Final dataset shape: (7536, 53)

Duplicate participant IDs:
0

Missing physical activity values:


PAD790Q      12
PAD790U       0
PAD800     1649
PAD810Q       9
PAD810U       0
PAD820     4168
PAD680       10
dtype: int64


Merged dataset preview:


,SEQN,RIDAGEYR,DIABETES,PAD790Q,PAD790U,PAD800,PAD810Q,PAD810U,PAD820,PAD680
0,130378.0,43.0,0,3.000000e+00,b'W',45.0,3.000000e+00,b'W',45.0,360.0
1,130379.0,66.0,0,4.000000e+00,b'W',45.0,3.000000e+00,b'W',45.0,480.0
2,130380.0,44.0,1,1.000000e+00,b'W',20.0,5.397605e-79,b'',NaN,240.0
3,130384.0,43.0,0,5.397605e-79,b'',NaN,5.397605e-79,b'',NaN,60.0
4,130385.0,65.0,0,1.000000e+00,b'D',90.0,1.000000e+00,b'W',60.0,180.0


### 6.5 Smoking Questionnaire Data (SMQ_L)

This section examines smoking-related information collected through the NHANES questionnaire.

The dataset will be inspected to identify relevant variables related to participants' smoking history and smoking behavior. The participant identifier (SEQN) will be used to integrate the selected variables into the existing adult diabetes dataset.

In [14]:
# Load smoking questionnaire dataset
smq_df = pd.read_sas(RAW_DIR / "SMQ_L.XPT", format="xport")

print("SMOKING QUESTIONNAIRE DATA")
print("=" * 50)

print("Dataset shape:", smq_df.shape)

print("\nAvailable columns:")
print(smq_df.columns.tolist())

print("\nMissing values:")
display(smq_df.isna().sum())

print("\nDuplicate participant IDs:")
print(smq_df["SEQN"].duplicated().sum())

print("\nDataset preview:")
display(smq_df.head())

SMOKING QUESTIONNAIRE DATA
Dataset shape: (9015, 9)

Available columns:
['SEQN', 'SMQ020', 'SMQ040', 'SMD641', 'SMD650', 'SMD100MN', 'SMQ621', 'SMD630', 'SMAQUEX2']

Missing values:


SEQN           0
SMQ020       880
SMQ040      5772
SMD641      8742
SMD650      7830
SMD100MN    7840
SMQ621      8243
SMD630      8992
SMAQUEX2       0
dtype: int64


Duplicate participant IDs:
0

Dataset preview:


,SEQN,SMQ020,SMQ040,SMD641,SMD650,SMD100MN,SMQ621,SMD630,SMAQUEX2
0,130378.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,1.0
1,130379.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,1.0
2,130380.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
3,130384.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,130385.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0


### 6.5.1 Merging Smoking Questionnaire Data

The smoking questionnaire variables are merged with the existing adult diabetes dataset using the participant identifier (SEQN).

A left join is used to preserve all participants. Missing smoking-related responses are retained for further investigation during the data-cleaning stage.

In [15]:
# Select smoking questionnaire variables
smq_selected = smq_df.copy()

# Merge smoking data with the existing dataset
adult_diabetes_smq_df = pd.merge(
    adult_diabetes_paq_df,
    smq_selected,
    on="SEQN",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("SMOKING QUESTIONNAIRE MERGE SUMMARY")
print("=" * 55)

print("Original participants:", len(adult_diabetes_paq_df))
print("Merged participants:", len(adult_diabetes_smq_df))

print("\nMerge status:")
print(adult_diabetes_smq_df["_merge"].value_counts())

# Remove merge indicator
adult_diabetes_smq_df.drop(columns=["_merge"], inplace=True)

print("\nFinal dataset shape:", adult_diabetes_smq_df.shape)

print("\nDuplicate participant IDs:")
print(adult_diabetes_smq_df["SEQN"].duplicated().sum())

print("\nMissing smoking questionnaire values:")
display(adult_diabetes_smq_df[smq_selected.columns.drop("SEQN")].isna().sum())

print("\nMerged dataset preview:")
display(adult_diabetes_smq_df[
    ["SEQN", "RIDAGEYR", "DIABETES",
     "SMQ020", "SMQ040", "SMD641", "SMD650",
     "SMD100MN", "SMQ621", "SMD630", "SMAQUEX2"]
].head())

SMOKING QUESTIONNAIRE MERGE SUMMARY
Original participants: 7536
Merged participants: 7536

Merge status:
_merge
both          7536
left_only        0
right_only       0
Name: count, dtype: int64

Final dataset shape: (7536, 61)

Duplicate participant IDs:
0

Missing smoking questionnaire values:


SMQ020        13
SMQ040      4443
SMD641      7298
SMD650      6407
SMD100MN    6407
SMQ621      7536
SMD630      7536
SMAQUEX2       0
dtype: int64


Merged dataset preview:


,SEQN,RIDAGEYR,DIABETES,SMQ020,SMQ040,SMD641,SMD650,SMD100MN,SMQ621,SMD630,SMAQUEX2
0,130378.0,43.0,0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,1.0
1,130379.0,66.0,0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,1.0
2,130380.0,44.0,1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
3,130384.0,43.0,0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,130385.0,65.0,0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0


### 6.6 Alcohol Use Questionnaire Data (ALQ_L)

This section examines alcohol consumption information collected through the NHANES questionnaire.

The dataset will be inspected to identify relevant variables related to alcohol consumption patterns and drinking frequency. The participant identifier (SEQN) will be used to integrate the selected variables into the existing adult diabetes dataset.

In [16]:
# Load alcohol use questionnaire dataset
alq_df = pd.read_sas(RAW_DIR / "ALQ_L.XPT", format="xport")

print("ALCOHOL USE QUESTIONNAIRE DATA")
print("=" * 50)

print("Dataset shape:", alq_df.shape)

print("\nAvailable columns:")
print(alq_df.columns.tolist())

print("\nMissing values:")
display(alq_df.isna().sum())

print("\nDuplicate participant IDs:")
print(alq_df["SEQN"].duplicated().sum())

print("\nDataset preview:")
display(alq_df.head())

ALCOHOL USE QUESTIONNAIRE DATA
Dataset shape: (6337, 9)

Available columns:
['SEQN', 'ALQ111', 'ALQ121', 'ALQ130', 'ALQ142', 'ALQ270', 'ALQ280', 'ALQ151', 'ALQ170']

Missing values:


SEQN         0
ALQ111     856
ALQ121    1415
ALQ130    2268
ALQ142    2255
ALQ270    3971
ALQ280    3975
ALQ151    1436
ALQ170    3979
dtype: int64


Duplicate participant IDs:
0

Dataset preview:


,SEQN,ALQ111,ALQ121,ALQ130,ALQ142,ALQ270,ALQ280,ALQ151,ALQ170
0,130378.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,130379.0,1.0,2.000000e+00,3.0,5.397605e-79,NaN,NaN,2.0,NaN
2,130380.0,1.0,1.000000e+01,1.0,5.397605e-79,NaN,NaN,2.0,NaN
3,130386.0,1.0,4.000000e+00,2.0,1.000000e+01,5.397605e-79,10.0,2.0,5.397605e-79
4,130387.0,1.0,5.397605e-79,NaN,NaN,NaN,NaN,2.0,NaN


### 6.6.1 Merging Alcohol Use Questionnaire Data

The alcohol use questionnaire variables are merged with the existing adult diabetes dataset using the participant identifier (SEQN).

A left join is used to preserve all participants. Missing alcohol consumption responses are retained for further investigation during the data-cleaning stage.

In [17]:
# Select alcohol use questionnaire variables
alq_selected = alq_df.copy()

# Merge alcohol data with the existing dataset
adult_diabetes_alq_df = pd.merge(
    adult_diabetes_smq_df,
    alq_selected,
    on="SEQN",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("ALCOHOL QUESTIONNAIRE MERGE SUMMARY")
print("=" * 55)

print("Original participants:", len(adult_diabetes_smq_df))
print("Merged participants:", len(adult_diabetes_alq_df))

print("\nMerge status:")
print(adult_diabetes_alq_df["_merge"].value_counts())

# Remove merge indicator
adult_diabetes_alq_df.drop(columns=["_merge"], inplace=True)

print("\nFinal dataset shape:", adult_diabetes_alq_df.shape)

print("\nDuplicate participant IDs:")
print(adult_diabetes_alq_df["SEQN"].duplicated().sum())

print("\nMissing alcohol questionnaire values:")
display(adult_diabetes_alq_df[alq_selected.columns.drop("SEQN")].isna().sum())

print("\nMerged dataset preview:")
display(adult_diabetes_alq_df[
    ["SEQN", "RIDAGEYR", "DIABETES",
     "ALQ111", "ALQ121", "ALQ130", "ALQ142",
     "ALQ270", "ALQ280", "ALQ151", "ALQ170"]
].head())

ALCOHOL QUESTIONNAIRE MERGE SUMMARY
Original participants: 7536
Merged participants: 7536

Merge status:
_merge
both          5843
left_only     1693
right_only       0
Name: count, dtype: int64

Final dataset shape: (7536, 69)

Duplicate participant IDs:
0

Missing alcohol questionnaire values:


ALQ111    2477
ALQ121    2925
ALQ130    3725
ALQ142    3715
ALQ270    5331
ALQ280    5335
ALQ151    2944
ALQ170    5339
dtype: int64


Merged dataset preview:


,SEQN,RIDAGEYR,DIABETES,ALQ111,ALQ121,ALQ130,ALQ142,ALQ270,ALQ280,ALQ151,ALQ170
0,130378.0,43.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,130379.0,66.0,0,1.0,2.0,3.0,5.397605e-79,NaN,NaN,2.0,NaN
2,130380.0,44.0,1,1.0,10.0,1.0,5.397605e-79,NaN,NaN,2.0,NaN
3,130384.0,43.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130385.0,65.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 6.7 Final Integrated Dataset Verification

This section performs a final verification of the integrated NHANES dataset.

The verification checks the dataset dimensions, participant identifiers, diabetes target distribution, duplicate records, and missing values.

The integrated dataset will be retained as the collected dataset for the subsequent data-cleaning and exploratory data analysis stages.

In [18]:
# Final integrated dataset
final_dataset = adult_diabetes_alq_df.copy()

print("FINAL INTEGRATED NHANES DATASET")
print("=" * 60)

print("\n1. Dataset dimensions:")
print("Rows:", final_dataset.shape[0])
print("Columns:", final_dataset.shape[1])

print("\n2. Duplicate participant IDs:")
print(final_dataset["SEQN"].duplicated().sum())

print("\n3. Diabetes target distribution:")
display(final_dataset["DIABETES"].value_counts().sort_index())

print("\n4. Diabetes target percentage:")
display(
    final_dataset["DIABETES"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\n5. Total missing values:")
print(final_dataset.isna().sum().sum())

print("\n6. Missing values by column:")
display(final_dataset.isna().sum().sort_values(ascending=False))

print("\n7. Dataset preview:")
display(final_dataset.head())

FINAL INTEGRATED NHANES DATASET

1. Dataset dimensions:
Rows: 7536
Columns: 69

2. Duplicate participant IDs:
0

3. Diabetes target distribution:


DIABETES
0    6466
1    1070
Name: count, dtype: int64


4. Diabetes target percentage:


DIABETES
0    85.8
1    14.2
Name: proportion, dtype: float64


5. Total missing values:
179026

6. Missing values by column:


DMDHSEDZ    7536
DMDHRMAZ    7536
RIDAGEMN    7536
DMDHREDZ    7536
DMDHRAGZ    7536
            ... 
BPQ080         0
SDMVPSU        0
BPQ020         0
DIABETES       0
SEQN           0
Length: 69, dtype: int64


7. Dataset preview:


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,SMD630,SMAQUEX2,ALQ111,ALQ121,ALQ130,ALQ142,ALQ270,ALQ280,ALQ151,ALQ170
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,NaN,1.0,1.0,2.0,3.0,5.397605e-79,NaN,NaN,2.0,NaN
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,NaN,1.0,1.0,10.0,1.0,5.397605e-79,NaN,NaN,2.0,NaN
3,130384.0,12.0,1.0,1.0,43.0,NaN,1.0,1.0,NaN,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130385.0,12.0,1.0,2.0,65.0,NaN,3.0,3.0,NaN,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 6.8 Saving the Integrated NHANES Dataset

The final integrated NHANES dataset is saved in CSV format in the data/processed directory.

This dataset contains demographic information, diabetes status, body measurements, blood pressure, physical activity, smoking, and alcohol consumption variables.

The saved dataset will serve as the input for the subsequent data-cleaning and preprocessing stage.

In [19]:
# Save the final integrated dataset

output_file = PROCESSED_DIR / "nhanes_diabetes_integrated.csv"

final_dataset.to_csv(output_file, index=False)

print("DATASET SAVED SUCCESSFULLY")
print("=" * 50)

print("File location:", output_file)
print("Dataset shape:", final_dataset.shape)
print("File exists:", output_file.exists())
print("File size (KB):", round(output_file.stat().st_size / 1024, 2))

DATASET SAVED SUCCESSFULLY
File location: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970/data/processed/nhanes_diabetes_integrated.csv
Dataset shape: (7536, 69)
File exists: True
File size (KB): 2057.31
